# Moonshine Streaming automatic speech recognition with OpenVINO

[Moonshine Streaming Medium](https://huggingface.co/moonshine-ai/moonshine-streaming-medium) is a 245M-parameter English ASR model with a sliding-window encoder and an autoregressive decoder. This notebook runs the official PyTorch model, converts it to a dynamic OpenVINO IR, validates transcription quality, and demonstrates NNCF INT8 weight-only compression.

The notebook uses the clean released OpenVINO path only. It does **not** rely on experimental compiler transformations or custom GPU kernels.

#### Table of contents:

1. [Prerequisites](#Prerequisites)
2. [Install Python packages](#Install-Python-packages)
3. [Prepare the Moonshine model and LibriSpeech sample](#Prepare-the-Moonshine-model-and-LibriSpeech-sample)
4. [Optional PyTorch XPU baseline](#Optional-PyTorch-XPU-baseline)
5. [Export the model to OpenVINO format](#Export-the-model-to-OpenVINO-format)
6. [Validation](#Validation)
    - [WER analysis](#WER-analysis)
    - [PyTorch XPU and OpenVINO logit similarity](#PyTorch-XPU-and-OpenVINO-logit-similarity)
7. [Inference with OpenVINO](#Inference-with-OpenVINO)
    - [Fixed-shape dynamic and static latency](#Fixed-shape-dynamic-and-static-latency)
8. [Quantization](#Quantization)
    - [Compress weights to INT8](#Compress-weights-to-INT8)
    - [Accuracy and model-size comparison](#Accuracy-and-model-size-comparison)
9. [OpenVINO inference with the quantized model](#OpenVINO-inference-with-the-quantized-model)
10. [Static and dynamic latency difference](#Static-and-dynamic-latency-difference)
11. [Findings](#Findings)

### Installation Instructions

Run this notebook in a Python virtual environment with Jupyter installed. The next cell installs the required packages. See the [OpenVINO Notebooks Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide) for details.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/moonshine-streaming-asr/moonshine-streaming-asr.ipynb" />

## Prerequisites

The [Moonshine Streaming Medium checkpoint](https://huggingface.co/moonshine-ai/moonshine-streaming-medium) is MIT-licensed. This notebook optionally evaluates on [LibriSpeech](https://www.openslr.org/12/), which is licensed under CC BY 4.0. Both licenses permit the model and evaluation workflow used here.

For comparable WER values, use the same audio, tokenizer, greedy decoding policy, maximum length `floor(valid_samples * 6.5 / 16000)`, and normalization: lowercase, remove punctuation, collapse whitespace, then compare word tokens. The supplied model card's benchmark score is not directly interchangeable with a local result because the evaluation protocol can differ.

[back to top ⬆️](#Moonshine-Streaming-automatic-speech-recognition-with-OpenVINO)


## Install Python packages

The Moonshine Streaming model class is available in recent Transformers releases. The CPU PyTorch wheel is sufficient for conversion and the portable CPU path. If an Intel XPU PyTorch build is already installed, the optional XPU section uses it without replacing it.


In [ ]:
%pip install -q "openvino==2026.4.1" "nncf==3.4.0" "transformers==5.21.0" "datasets[audio]==5.1.0" jiwer soundfile torch

import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    response = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        timeout=30,
    )
    response.raise_for_status()
    Path("notebook_utils.py").write_text(response.text, encoding="utf-8")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("moonshine-streaming-asr.ipynb")

In [ ]:
from __future__ import annotations

import io
import statistics
import time
from pathlib import Path

import jiwer
import nncf
import numpy as np
import openvino as ov
import soundfile as sf
import torch
from datasets import Audio, load_dataset
from transformers import AutoProcessor, MoonshineStreamingForConditionalGeneration

MODEL_ID = "moonshine-ai/moonshine-streaming-medium"
SAMPLE_RATE = 16_000
MODEL_DIR = Path("model").resolve()
MODEL_DIR.mkdir(exist_ok=True)

print(f"OpenVINO: {ov.get_version()}")
print(f"NNCF: {nncf.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch XPU available: {hasattr(torch, 'xpu') and torch.xpu.is_available()}")


## Prepare the Moonshine model and LibriSpeech sample

Moonshine expects raw 16 kHz waveform samples, a binary attention mask, and growing decoder token IDs. The input preparation below pads waveform and mask together to the model's 320-sample alignment.


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
torch_model = MoonshineStreamingForConditionalGeneration.from_pretrained(MODEL_ID, attn_implementation="sdpa").eval()

dataset = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)
dataset = dataset.cast_column("audio", Audio(decode=False))
sample = next(iter(dataset))
audio_source = sample["audio"]
if audio_source.get("bytes") is not None:
    audio, audio_sample_rate = sf.read(io.BytesIO(audio_source["bytes"]), dtype="float32")
else:
    audio, audio_sample_rate = sf.read(audio_source["path"], dtype="float32")
if audio_sample_rate != SAMPLE_RATE:
    raise ValueError(f"Expected {SAMPLE_RATE} Hz LibriSpeech audio, got {audio_sample_rate} Hz")
if audio.ndim == 2:
    audio = audio.mean(axis=1)
audio = np.asarray(audio, dtype=np.float32)
reference = str(sample["text"])


def prepare_inputs(audio_array: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:
    """Prepare aligned waveform and int64 mask tensors for the OpenVINO model.

    :param audio_array: Mono 16 kHz float32 waveform.
    :returns: Padded waveform, padded attention mask, and valid sample count.
    """
    processed = processor(audio_array, sampling_rate=SAMPLE_RATE, return_tensors="np")
    waveform = np.asarray(processed["input_values"][0], dtype=np.float32)
    mask = np.asarray(processed["attention_mask"][0], dtype=np.int64)
    if waveform.shape != mask.shape or not np.isin(mask, (0, 1)).all():
        raise ValueError("Moonshine processor must return matching binary waveform and mask shapes")
    padded_length = max(1600, ((waveform.size + 319) // 320) * 320)
    return np.pad(waveform, (0, padded_length - waveform.size)), np.pad(mask, (0, padded_length - mask.size)), int(mask.sum())


waveform, attention_mask, valid_samples = prepare_inputs(audio)
max_length = int(valid_samples * 6.5 / SAMPLE_RATE)
print(f"Reference: {reference}")
print(f"Waveform shape: {waveform.shape}; valid samples: {valid_samples}; max length: {max_length}")


## Optional PyTorch XPU baseline

This section verifies that the original Moonshine PyTorch model runs on a supported Intel GPU through PyTorch XPU. Its transcript is a source-model reference used later to compare quality and one-forward logits against the converted OpenVINO IR.

It is optional and requires a supported Intel GPU plus an XPU-enabled PyTorch build. It is not required for OpenVINO export, inference, WER evaluation, or quantization. Users without XPU support will see a skip message.

To enable it on a supported Intel GPU, install the tested packages, **restart the notebook kernel**, and rerun the notebook from the imports cell:

```bash
pip install "torch==2.14.1+xpu" "torchvision==0.29.1+xpu" --index-url https://download.pytorch.org/whl/xpu
```

In [ ]:
RUN_PYTORCH_XPU = True
XPU_AVAILABLE = hasattr(torch, "xpu") and torch.xpu.is_available()

if not RUN_PYTORCH_XPU:
    print("Skipped PyTorch XPU baseline. Set RUN_PYTORCH_XPU = True on a supported Intel GPU to run it.")
elif not XPU_AVAILABLE:
    print("Skipped PyTorch XPU baseline: torch.xpu.is_available() is False.")
else:
    xpu_model = torch_model.to("xpu").eval()
    inputs = {
        "input_values": torch.from_numpy(waveform[None]).to("xpu"),
        "attention_mask": torch.from_numpy(attention_mask[None]).to("xpu"),
        "max_length": max_length,
    }
    with torch.inference_mode():
        generated_ids = xpu_model.generate(**inputs)
    print("PyTorch XPU source-model inference completed successfully.")
    print(f"PyTorch XPU transcript: {processor.decode(generated_ids[0], skip_special_tokens=True)}")

## Export the model to OpenVINO format

The three public inputs must remain dynamic: audio length changes by utterance, and `decoder_input_ids` grows from `[1, 1]` to `[1, N]` during autoregressive decoding.


In [ ]:
fp16_ir_path = MODEL_DIR / "moonshine-streaming-medium-fp16.xml"

class LogitsForward(torch.nn.Module):
    """Export only the no-cache forward logits needed by growing-prefix decoding."""

    def __init__(self, model: MoonshineStreamingForConditionalGeneration) -> None:
        super().__init__()
        self.model = model

    def forward(
        self, input_values: torch.Tensor, attention_mask: torch.Tensor, decoder_input_ids: torch.Tensor
    ) -> torch.Tensor:
        return self.model(
            input_values=input_values,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            use_cache=False,
        ).logits


if not fp16_ir_path.exists():
    # This is the validated no-cache export path from export_openvino.py.
    # Eager attention is used only while exporting; the PyTorch XPU baseline above uses SDPA.
    export_model = MoonshineStreamingForConditionalGeneration.from_pretrained(
        MODEL_ID, attn_implementation="eager", dtype=torch.float32
    ).eval()
    sample_count = int(export_model.config.encoder_config.sample_rate)
    export_input_values = torch.zeros(1, sample_count, dtype=torch.float32)
    export_attention_mask = torch.ones(1, sample_count, dtype=torch.long)
    export_decoder_ids = torch.full((1, 3), export_model.config.decoder_start_token_id, dtype=torch.long)
    audio_blocks = 320 * torch.export.Dim("audio_blocks", min=5)
    exported = torch.export.export(
        LogitsForward(export_model).eval(),
        (export_input_values, export_attention_mask, export_decoder_ids),
        dynamic_shapes={
            "input_values": {1: audio_blocks},
            "attention_mask": {1: audio_blocks},
            "decoder_input_ids": {1: torch.export.Dim("tokens", min=2, max=256)},
        },
    )
    ov_model = ov.convert_model(exported)
    ov.save_model(ov_model, fp16_ir_path, compress_to_fp16=True)
    fp16_bin_path = fp16_ir_path.with_suffix(".bin")
    if not fp16_bin_path.is_file():
        raise RuntimeError(f"OpenVINO export did not create weights: {fp16_bin_path}")
    print("Generated OpenVINO FP16 IR:")
    print(f"  XML: {fp16_ir_path}")
    print(f"  BIN: {fp16_bin_path} ({fp16_bin_path.stat().st_size / 1024**2:.1f} MiB)")
else:
    ov_model = ov.Core().read_model(fp16_ir_path)
    print("Using existing OpenVINO FP16 IR:")
    print(f"  XML: {fp16_ir_path}")
    print(f"  BIN: {fp16_ir_path.with_suffix('.bin')}")

for input_port in ov_model.inputs:
    print(f"{input_port.any_name}: {input_port.get_partial_shape()} {input_port.get_element_type()}")


## Validation

### WER analysis

Set `RUN_WER_EVALUATION` to `True` to evaluate the dynamic FP16 OpenVINO IR on the first 10 LibriSpeech `test-clean` utterances in source order. The implementation uses the same input padding, greedy growing-prefix decoding, maximum-length policy, and JiWER normalization as the local evaluation workflow.

The following 500-utterance results provide the larger matched quality comparison. They were measured on an Intel Arc B390 GPU with OpenVINO 2026.3.1.

| Runtime / IR | Utterances | WER |
| --- | ---: | ---: |
| PyTorch XPU, official cached generation | 500 | 2.5988% |
| OpenVINO FP16 dynamic IR | 500 | 2.5896% |



In [ ]:
def generate_openvino(
    compiled_model: ov.CompiledModel,
    waveform: np.ndarray,
    mask: np.ndarray,
    max_length: int,
    bos_token_id: int,
    eos_token_id: int,
) -> tuple[list[int], float]:
    """Generate greedy Moonshine token IDs with a growing decoder prefix."""
    request = compiled_model.create_infer_request()
    tokens = [bos_token_id]
    started = time.perf_counter()
    for _ in range(max_length - 1):
        logits = request.infer(
            {
                "input_values": waveform[None],
                "attention_mask": mask[None],
                "decoder_input_ids": np.asarray([tokens], dtype=np.int64),
            }
        )[compiled_model.output(0)]
        next_token = int(np.argmax(logits[0, -1]))
        tokens.append(next_token)
        if next_token == eos_token_id:
            break
    return tokens, (time.perf_counter() - started) * 1000


core = ov.Core()
DEVICE = "GPU" if "GPU" in core.available_devices else "CPU"
compiled_model = core.compile_model(ov_model, DEVICE)
bos_token_id = int(processor.tokenizer.bos_token_id)
eos_token_id = int(processor.tokenizer.eos_token_id)

In [ ]:
RUN_WER_EVALUATION = True
WER_LIMIT = 10
NORMALIZE = jiwer.Compose(
    [jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip(), jiwer.ReduceToListOfListOfWords()]
)

if RUN_WER_EVALUATION:
    references, hypotheses = [], []
    for index, row in enumerate(dataset.take(WER_LIMIT), start=1):
        evaluation_audio_source = row["audio"]
        if evaluation_audio_source.get("bytes") is not None:
            evaluation_audio, evaluation_sample_rate = sf.read(
                io.BytesIO(evaluation_audio_source["bytes"]), dtype="float32"
            )
        else:
            evaluation_audio, evaluation_sample_rate = sf.read(
                evaluation_audio_source["path"], dtype="float32"
            )
        if evaluation_sample_rate != SAMPLE_RATE:
            raise ValueError(f"Expected {SAMPLE_RATE} Hz LibriSpeech audio, got {evaluation_sample_rate} Hz")
        if evaluation_audio.ndim == 2:
            evaluation_audio = evaluation_audio.mean(axis=1)
        evaluation_waveform, evaluation_mask, evaluation_samples = prepare_inputs(
            np.asarray(evaluation_audio, dtype=np.float32)
        )
        evaluation_tokens, _ = generate_openvino(
            compiled_model,
            evaluation_waveform,
            evaluation_mask,
            int(evaluation_samples * 6.5 / SAMPLE_RATE),
            bos_token_id,
            eos_token_id,
        )
        references.append(str(row["text"]))
        hypotheses.append(processor.decode(evaluation_tokens, skip_special_tokens=True))
        print(f"Scored {index}/{WER_LIMIT}", end="\r")
    measure = jiwer.process_words(references, hypotheses, reference_transform=NORMALIZE, hypothesis_transform=NORMALIZE)
    print(f"\nWER: {measure.wer:.4%}; S/D/I: {measure.substitutions}/{measure.deletions}/{measure.insertions}")


### Optional PyTorch XPU and OpenVINO logit similarity

This optional numerical check runs the same one-token decoder forward on PyTorch XPU and the OpenVINO FP16 IR. It uses identical prepared LibriSpeech audio, attention mask, and beginning-of-sequence token, then reports cosine similarity, MSE, RMSE, maximum absolute error, and greedy next-token agreement.

It requires a supported Intel GPU and the XPU PyTorch packages shown in the preceding section. Users without XPU support receive a skip message; the rest of the notebook remains runnable. This is a fixed-forward conversion check, not an end-to-end autoregressive latency comparison.

In [ ]:
RUN_XPU_LOGIT_COMPARISON = True

if not RUN_XPU_LOGIT_COMPARISON:
    print("Skipped PyTorch XPU/OpenVINO logit comparison.")
elif not XPU_AVAILABLE:
    print(
        "Skipped PyTorch XPU/OpenVINO logit comparison: torch.xpu.is_available() is False. "
        "Install the optional XPU PyTorch packages on a supported Intel GPU, then restart the kernel."
    )
else:
    xpu_model = torch_model.to("xpu").eval()
    xpu_input_values = torch.from_numpy(waveform[None]).to("xpu")
    xpu_attention_mask = torch.from_numpy(attention_mask[None]).to("xpu")
    xpu_decoder_ids = torch.as_tensor([[bos_token_id]], dtype=torch.long, device="xpu")

    with torch.inference_mode():
        torch.xpu.synchronize()
        pytorch_xpu_logits = xpu_model(
            input_values=xpu_input_values,
            attention_mask=xpu_attention_mask,
            decoder_input_ids=xpu_decoder_ids,
            use_cache=False,
        ).logits.float().cpu().numpy()
        torch.xpu.synchronize()

    openvino_logits = compiled_model(
        {
            "input_values": waveform[None],
            "attention_mask": attention_mask[None],
            "decoder_input_ids": np.asarray([[bos_token_id]], dtype=np.int64),
        }
    )[compiled_model.output(0)]

    if pytorch_xpu_logits.shape != openvino_logits.shape:
        raise ValueError(
            "PyTorch XPU and OpenVINO logits must have identical shapes; "
            f"got {pytorch_xpu_logits.shape} and {openvino_logits.shape}"
        )

    pytorch_xpu_logits = pytorch_xpu_logits.astype(np.float64, copy=False)
    openvino_logits = openvino_logits.astype(np.float64, copy=False)
    difference = pytorch_xpu_logits - openvino_logits
    cosine_similarity = np.dot(pytorch_xpu_logits.ravel(), openvino_logits.ravel()) / (
        np.linalg.norm(pytorch_xpu_logits.ravel()) * np.linalg.norm(openvino_logits.ravel())
    )
    pytorch_next_token = int(np.argmax(pytorch_xpu_logits[0, -1]))
    openvino_next_token = int(np.argmax(openvino_logits[0, -1]))

    print(f"Logits shape: {pytorch_xpu_logits.shape}")
    print(f"MSE: {np.mean(difference ** 2):.8g}")
    print(f"RMSE: {np.sqrt(np.mean(difference ** 2)):.8g}")
    print(f"Maximum absolute error: {np.max(np.abs(difference)):.8g}")
    print(f"Cosine similarity: {cosine_similarity:.10f}")
    print(f"PyTorch XPU next token: {pytorch_next_token}")
    print(f"OpenVINO next token:    {openvino_next_token}")
    print(f"Next-token agreement: {pytorch_next_token == openvino_next_token}")

For the first LibriSpeech `test-clean` sample on Intel Arc B390, this comparison measured the following with the released dynamic FP16 IR:

| Metric | Result |
| --- | ---: |
| Logit shape | `[1, 1, 32768]` |
| MSE | `3.0503597e-05` |
| RMSE | `0.0055230062` |
| Maximum absolute error | `0.027698517` |
| Cosine similarity | `0.9999995829` |
| Greedy next-token agreement | `True` |



## Inference with OpenVINO

A call with decoder shape `[1, 1]` predicts only the next token; it does not transcribe the whole clip. The helper below feeds each predicted token back into the next call, changing the decoder input shape for every step.


### Dynamic and static shape latency

This measures the same unit of work as `benchmark_app`: one repeated, fixed-shape forward pass. It does not generate a complete transcript. The baseline below uses the long representative shape `input_values=[1,200000]`, `attention_mask=[1,200000]`, and `decoder_input_ids=[1,1]`.

It compiles two separate models with identical tensors: the original dynamic IR and a static specialization. To compare Python and `benchmark_app` numbers, use the same device, shapes, performance configuration, warmup, and iterations.


In [ ]:
def benchmark_fixed_forward(
    compiled_model: ov.CompiledModel,
    waveform: np.ndarray,
    mask: np.ndarray,
    decoder_token_id: int,
    warmup_iterations: int = 10,
    measured_iterations: int = 100,
) -> list[float]:
    """Measure repeated inference for one fixed decoder-prefix shape.

    :param compiled_model: Compiled dynamic Moonshine OpenVINO model.
    :param waveform: Padded float32 waveform with shape [samples].
    :param mask: Padded int64 attention mask with shape [samples].
    :param decoder_token_id: The one-token decoder prefix value.
    :param warmup_iterations: Untimed inferences before measurement.
    :param measured_iterations: Number of synchronous inferences to time.
    :returns: Per-inference latencies in milliseconds.
    """
    request = compiled_model.create_infer_request()
    request.set_tensor("input_values", ov.Tensor(waveform[None]))
    request.set_tensor("attention_mask", ov.Tensor(mask[None]))
    request.set_tensor("decoder_input_ids", ov.Tensor(np.asarray([[decoder_token_id]], dtype=np.int64)))
    for _ in range(warmup_iterations):
        request.infer()
    latencies_ms = []
    for _ in range(measured_iterations):
        started = time.perf_counter()
        request.infer()
        latencies_ms.append((time.perf_counter() - started) * 1000)
    return latencies_ms


BENCHMARK_AUDIO_SAMPLES = 200_000
benchmark_waveform = np.zeros(BENCHMARK_AUDIO_SAMPLES, dtype=np.float32)
benchmark_attention_mask = np.ones(BENCHMARK_AUDIO_SAMPLES, dtype=np.int64)

# Keep this IR dynamic at compile time, then repeatedly infer one concrete shape.
dynamic_benchmark_model = core.read_model(fp16_ir_path)
dynamic_benchmark_compiled = core.compile_model(dynamic_benchmark_model, DEVICE)
dynamic_forward_latencies_ms = benchmark_fixed_forward(
    dynamic_benchmark_compiled, benchmark_waveform, benchmark_attention_mask, bos_token_id
)

# Specialize every public input before compilation for a static-kernel comparison.
static_benchmark_model = core.read_model(fp16_ir_path)
static_benchmark_model.reshape(
    {
        "input_values": [1, BENCHMARK_AUDIO_SAMPLES],
        "attention_mask": [1, BENCHMARK_AUDIO_SAMPLES],
        "decoder_input_ids": [1, 1],
    }
)
static_benchmark_compiled = core.compile_model(static_benchmark_model, DEVICE)
static_forward_latencies_ms = benchmark_fixed_forward(
    static_benchmark_compiled, benchmark_waveform, benchmark_attention_mask, bos_token_id
)

print(f"Fixed forward shapes: audio/mask [1, {BENCHMARK_AUDIO_SAMPLES}], decoder IDs [1, 1]")
print(f"Dynamic IR median: {statistics.median(dynamic_forward_latencies_ms):.2f} ms")
print(f"Static IR median:  {statistics.median(static_forward_latencies_ms):.2f} ms")


## Quantization

### Compress weights to INT8

NNCF INT8 weight-only compression is data-free: it compresses eligible weights but does not calibrate activations or consume LibriSpeech examples. Reload the original FP16 IR before compression so it remains the unmodified accuracy and latency reference.

In [ ]:
int8_ir_path = MODEL_DIR / "moonshine-streaming-medium-int8-weights.xml"

if not int8_ir_path.exists():
    int8_model = nncf.compress_weights(
        core.read_model(fp16_ir_path), mode=nncf.CompressWeightsMode.INT8_ASYM
    )
    ov.save_model(int8_model, int8_ir_path, compress_to_fp16=False)

fp16_size_mib = fp16_ir_path.with_suffix(".bin").stat().st_size / 1024**2
int8_size_mib = int8_ir_path.with_suffix(".bin").stat().st_size / 1024**2
print(f"FP16 weights: {fp16_size_mib:.1f} MiB")
print(f"INT8 weights: {int8_size_mib:.1f} MiB")
print(f"Reduction: {(1 - int8_size_mib / fp16_size_mib):.1%}")

### Accuracy and model-size comparison

| IR | Weight file size | LibriSpeech `test-clean` WER (500 utterances) | Change vs. FP16 |
| --- | ---: | ---: | ---: |
| FP16 dynamic IR | 489.9 MiB | 2.5896% | baseline |
| NNCF INT8 asymmetric weight-only IR | 246.0 MiB | 2.7566% | +0.1670 percentage points; 18 extra word errors |

INT8 reduces the weight file by about 50%, with a small but measurable quality regression on the matched 500-utterance subset. Define an explicit WER tolerance and evaluate a larger corpus before using it as a deployment default.

## OpenVINO inference with the quantized model

Compile the INT8 dynamic IR through the same OpenVINO Python API. This repeats the same fixed forward used for the FP16 dynamic baseline: audio and mask `[1,200000]`, decoder IDs `[1,1]`, ten warmups, and 100 measured inferences.

In [ ]:
int8_benchmark_compiled = core.compile_model(core.read_model(int8_ir_path), DEVICE)
int8_forward_latencies_ms = benchmark_fixed_forward(
    int8_benchmark_compiled,
    benchmark_waveform,
    benchmark_attention_mask,
    bos_token_id,
)
print(f"INT8 dynamic IR median: {statistics.median(int8_forward_latencies_ms):.2f} ms")
print(f"FP16 dynamic IR median: {statistics.median(dynamic_forward_latencies_ms):.2f} ms")
print(f"FP16 static IR median:  {statistics.median(static_forward_latencies_ms):.2f} ms")

## Static and dynamic latency difference

For the controlled `[1,200000]` audio/mask and `[1,1]` decoder forward on the current GPU, the released FP16 IR measured a **22.92 ms dynamic median** and an **11.34 ms static median**. Re-run the preceding benchmark cell on your system because device, driver, runtime, and configuration affect the absolute values.

Static specialization is useful as a roofline-oriented lower-bound baseline, but it is not an ideal ASR deployment shape. Real utterances have variable audio lengths, and Moonshine autoregressive decoding grows from `[1,1]` to `[1,2]`, `[1,3]`, and beyond. A static graph would require multiple shape-specialized compilations, padding waste, or recompilation as the prefix grows. The deployable path therefore needs a performant **dynamic-shape graph**, then must be optimized and validated at representative dynamic shapes.

## Findings

### Fusion and dynamic-shape opportunities

- **SDPA fusion:** The original dynamic graph contains decomposed attention chains: `MatMul(Q, K^T) → scale → Add(mask) → Softmax → Broadcast → MatMul(V)`. The post-Softmax `Broadcast` prevented the expected dynamic SDPA pattern from fusing in the investigated graph. Any fusion must preserve Moonshine's mask semantics and be numerically checked before use.
- **RoPE fusion:** Moonshine uses partial interleaved RoPE: it rotates the first 32 dimensions of each 64-wide head and preserves the remaining 32 dimensions. A generic full-head RoPE fusion is therefore not equivalent; an eligible fusion needs this exact partial-rotation behavior and validation.
- **Dynamic ReduceSum:** The dynamic attention-mask reduction (`attention_mask [B,L] → ReduceSum(axis=-1)`) selected the GPU reference implementation `reduce_ref__i32` in profiling and was a material hotspot. Static specialization can use a cheaper scalar path, which contributes to the latency gap above. This makes dynamic reduction support a concrete optimization target.

### Static-shape roofline analysis

The roofline estimates below were collected for one static FP16 forward with audio/mask `[1,200000]` and decoder IDs `[1,1]` on the Intel Arc B390. They bound the best achievable time for this controlled forward only; they are not complete autoregressive transcription latency.

| Estimate | Latency (ms) | Interpretation |
| --- | ---: | --- |
| Ideal roofline lower bound | 8.960 | Hardware throughput and bandwidth bound, excluding launch overhead. |
| Device-launch-adjusted bound | 9.466 | Adds estimated GPU device-launch cost. |
| Host-launch-adjusted bound | 11.228 | Adds host-side launch cost; closest roofline comparison to the synchronous Python API measurement. |
| Measured static FP16 median | 11.344 | OpenVINO Python API, 10 warmups and 100 fixed-forward measurements. |

The 11.344 ms measured static median is close to the 11.228 ms host-launch-adjusted bound. This indicates that the static `[1,200000]` / `[1,1]` forward is already near its estimated roofline on this system. The 22.924 ms dynamic median therefore points to dynamic-shape execution behavior—not a general lack of static forward throughput—as the primary optimization opportunity.

### Conclusion

Moonshine Streaming Medium was successfully exported to a released dynamic OpenVINO FP16 IR and maintained recognition quality on the matched 500-utterance LibriSpeech comparison: **2.5896% OpenVINO FP16 WER** versus **2.5988% PyTorch XPU WER**. The optional XPU logit check also confirmed near-identical one-forward outputs and the same greedy next token.

The static fixed-forward result is close to its roofline bound, but static compilation is not a practical end-to-end ASR solution because audio duration and the autoregressive decoder prefix are variable. The next performance work should therefore target the released dynamic graph: profile matched dynamic shapes, eliminate material reference/reorder overhead, and evaluate semantically valid SDPA, partial-RoPE, and dynamic-ReduceSum improvements one at a time. Each candidate should be accepted only after execution-graph inspection, numerical/WER validation, and repeated latency measurements.